# ViMed-RAG — Tuần 2: embed + index lên Qdrant Cloud

Notebook này **mỏng có chủ đích**: mọi logic nằm trong repo (`src/`, `scripts/`),
ở đây chỉ là dây nối. Sửa logic thì sửa trong repo rồi commit, đừng sửa trong cell —
code trong notebook Kaggle không ai review được và không có test.

## Chuẩn bị TRƯỚC khi chạy (làm 1 lần)

1. **Dataset**: upload `data/processed/corpus.jsonl` (12,8 MB) thành Kaggle Dataset
   **private**. Add Data → chọn dataset đó. Sửa `CORPUS` ở cell 4 cho khớp slug.
   *Vì sao không chạy lại ingestion ở đây:* DEC-020 (4.972 chunk, 3 tiêu chí PASS)
   đo trên **đúng file này**. Sinh lại corpus = số của DEC-020 hết hiệu lực mà không ai biết.
2. **Secrets**: Add-ons → Secrets → thêm `QDRANT_URL` và `QDRANT_API_KEY`.
   **Không dán key vào cell** — notebook hay bị share/public.
3. **GPU**: Settings → Accelerator → **GPU T4 x2** (dùng 1 cái là đủ).

## Kiểm chéo sau khi chạy

| size | chunk (DEC-020) | collection |
|---|---|---|
| 512 | 4.972 | `vimed_rag_512` |
| 256 | 10.246 | `vimed_rag_256` |

Chạy lại **an toàn**: point ID tất định (UUID5 của `doc_id:chunk_idx`) nên upsert đè,
không nhân đôi. Session đứt giữa chừng → chạy lại nguyên lệnh, hoặc thêm
`--start <số point đã xong>` để khỏi embed lại phần đã có.

### 1. Cài dependency

`FlagEmbedding` có thể kéo theo torch bản khác bản Kaggle cài sẵn. Nếu cell dưới
báo cần restart thì **Run → Restart & Clear Cell Outputs** rồi chạy lại từ cell 2
(bỏ qua cell này).

In [ ]:
!pip install -q "FlagEmbedding>=1.2,<2.0" "qdrant-client>=1.9"
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

### 2. Lấy code

Repo public → `git clone` là xong. Nếu repo **private**: hoặc upload cả repo thành
một Kaggle Dataset nữa rồi `cp -r` vào `/kaggle/working/vimed-rag`, hoặc thêm
GitHub PAT vào Secrets và clone qua HTTPS có token.

In [ ]:
import os, shutil

REPO_DIR = "/kaggle/working/vimed-rag"
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)  # luôn lấy bản mới nhất, tránh chạy nhầm code cũ

!git clone --depth 1 https://github.com/dateddy/vimed-rag.git {REPO_DIR}
os.chdir(REPO_DIR)
!git log --oneline -1

### 3. Secrets → biến môi trường

`load_config()` đọc `${QDRANT_URL}` / `${QDRANT_API_KEY}` từ `os.environ`
(file `.env` không có trên Kaggle — đúng như mong muốn).

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["QDRANT_URL"] = secrets.get_secret("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = secrets.get_secret("QDRANT_API_KEY")

# In có che — đủ để biết đã nạp đúng cluster, không lộ key.
print("URL :", os.environ["QDRANT_URL"][:40] + "...")
print("KEY :", "đã nạp" if os.environ["QDRANT_API_KEY"] else "RỖNG (sai tên secret?)")

!python scripts/smoke_qdrant.py

### 4. Trỏ tới corpus

Sửa `CORPUS` cho khớp slug dataset của bạn. Cell tự liệt kê `/kaggle/input` nếu sai đường dẫn.

In [ ]:
CORPUS = "/kaggle/input/vimed-rag-corpus/corpus.jsonl"  # <-- SỬA cho khớp slug

if not os.path.exists(CORPUS):
    print("!! Không thấy", CORPUS, "— các file đang có trong /kaggle/input:")
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            print("  ", os.path.join(root, f))
else:
    print("OK:", CORPUS, f"({os.path.getsize(CORPUS) / 1024**2:.1f} MB)")

# Kiểm đường ống trước khi đụng GPU: chunk + đếm, không kết nối Qdrant.
!python scripts/build_index.py --size 512 --corpus {CORPUS} --dry-run

### 5. Index chunk 512 (~4.972 point)

Lần chạy đầu tải bge-m3 (~2,3 GB) nên chậm hơn. Theo dõi dòng `chunk/s` + ETA.

In [ ]:
!python scripts/build_index.py --size 512 --corpus {CORPUS}

In [ ]:
!python scripts/verify_index.py --size 512

### 6. Index chunk 256 (~10.246 point) — ablation DEC-004

Chỉ chạy khi cell verify ở trên đã **PASS**.

In [ ]:
!python scripts/build_index.py --size 256 --corpus {CORPUS}

In [ ]:
!python scripts/verify_index.py --size 256

### 7. Ghi lại số thực để đóng Tuần 2

Chép output của cell dưới vào `brain/state/STATUS.md`. Ngân sách đã chốt ở DEC-016
là **~8% free tier 1GB cho cả 2 collection** — nếu số thực lệch xa thì phải ghi
một dòng DECISIONS mới, đừng để lệch âm thầm.

In [ ]:
from qdrant_client import QdrantClient

client = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"], timeout=60)
for c in client.get_collections().collections:
    print(f"{c.name:20s} {client.count(c.name, exact=True).count:7d} point")